In [49]:
def tag_dict_to_str(tag_dict):
    string = ""
    for key, val in tag_dict.items():
        string += key
        string += ': '
        string += str(val)
        string += ', '
    return string


def has_not_relevant_key(string, not_relevant_keys):
    split = [string]
    if ':' in string:
        split = string.split(':')
    elif '_' in string:
        split = string.split('_')
    for str in split:
        if str.lower() in not_relevant_keys:
            return True
    return False


not_relevant_keys = {'addr', 'comment', 'contact', 'source', 'name', 'tiger',
                     'ref', 'created_by', 'nysgissam', 'wikidata', 'operator',
                     'lacounty', 'osak', 'source_ref', 'nhd', 'admin_level',
                     'wikipedia', 'yh', 'gnis', 'at_bev', 'mml', 'postal_code',
                     'raba', 'nycdoitt', 'maaamet', 'pmfsefin', 'old_name', 'official_name',
                     'chicago', 'linz', 'it', 'destination', 'date', 'lojic'
                     'geobase', 'mapillary', 'clc', 'ssr', 'unsigned_ref', 'naptan'
                     'mvdgis', 'linz2osm', 'gns', 'note', 'metcouncil', 'url',
                     'route_ref', 'gtfs', 'uic', 'attribution', 'date', 'ts',
                     'id', 'survey', 'stif', 'network', 'naptan', 'location',
                     'tmc', 'fixme', 'wabe', 'object', 'description', 'check_date',
                     'tec', 'qroti', 'dcgis', 'website', 'short_name', 'image',
                     'NaPTANAreaCode', 'vrs', 'cxx', 'in', 'code', 'massgis', 'original_osm_id', 
                     'bbr', 'shape', 'lnam', 'redwood_city_ca', 'email', 'KSJ2',
                     'canvec', 'uuid', 'sorting_name', 'phone', 'inegi', 'ine', 
                     'brand', 'cesena', 'geobase', 'mobile', 'strazakosm', 'ipp',
                     'fhrs', 'alt_name', 'old_street', 'ksj2', 'unocha',
                     'wikimedia_commons', 'lojic', 'brn', 'fid', 'notas',
                     'fax', 'sangis', 'okato', 'nhd-shp', 'surrey', 'statscan',
                     'panoramax'}
                                                                 

repeat_no = 20

In [ ]:
import pickle
import gc

gc.disable()

with open('osm_planet_unique_relevant_tagsets.pkl', 'rb') as file:
    unique_relevant_tagsets = pickle.load(file)
    print("Loaded file " + file.name)

gc.enable()

In [ ]:
keys = list(unique_relevant_tagsets.keys())
print("Unique relevant tagsets: " + str(len(unique_relevant_tagsets)))

In [50]:
import random
dataset_dict = {'anchor': [], 'positive': []}
for _ in range(0, repeat_no):
    for el in unique_relevant_tagsets.keys():
        if len(unique_relevant_tagsets[el]) < _:
            continue
        idx_anchor = random.randrange(0, len(unique_relevant_tagsets[el]))
        idx_positive = random.randrange(
            0, len(unique_relevant_tagsets[el]))
        if idx_anchor != idx_positive:
            dataset_dict['anchor'].append(tag_dict_to_str(
                unique_relevant_tagsets[el][idx_anchor]))
            dataset_dict['positive'].append(tag_dict_to_str(
                unique_relevant_tagsets[el][idx_positive]))
        else:
            relevant_tags = {key: val for key, val in unique_relevant_tagsets[el][idx_positive].items(
            ) if not has_not_relevant_key(key, not_relevant_keys)}
            dataset_dict['anchor'].append(tag_dict_to_str(
                unique_relevant_tagsets[el][idx_anchor]))
            dataset_dict['positive'].append(tag_dict_to_str(relevant_tags))



In [ ]:
print(len(dataset_dict['anchor']))
print(len(dataset_dict['positive']))

gc.disable()

with open('relevant_tags_pairs_dataset_osm_planet_20_rep.pkl', 'wb') as handle:
    pickle.dump(dataset_dict, handle, protocol=pickle.HIGHEST_PROTOCOL) 

gc.enable()

In [ ]:
from datasets import Dataset
dataset = Dataset.from_dict(dataset_dict)

In [ ]:
num_shards = 128
output_path_template = "relevant_tags_pairs_dataset_parquet_sharded_20_rep/relevant_tags_pairs_dataset_osm_planet_20_rep_{index:05d}.parquet"
for index in range(num_shards):
    shard = dataset.shard(index=index, num_shards=num_shards, contiguous=True)
    shard.to_parquet(output_path_template.format(index=index))